![NVIDIA Logo](images/nvidia.png)

# Identity Inference Stage

In this notebook we showcase the use of an inference stage in a Morpheus pipeline, and build a custom stage to create and attach sequence ID tensors to the pipeline's control messages prior to inference.

---

## Objectives

By the time you complete this notebook you will be able to:

- Create custom stages that add tensors to control messages in preparation for downstream inference.
- Perform no-op inference in a Morpheus pipeline.

---

## Imports

In [ ]:
import typing
import logging

import cupy as cp

from IPython.display import Image

from morpheus.config import Config
from morpheus.pipeline import LinearPipeline

from morpheus.stages.general.monitor_stage import MonitorStage
from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.preprocess.deserialize_stage import DeserializeStage
from morpheus.stages.output.in_memory_sink_stage import InMemorySinkStage
from morpheus.stages.inference.identity_inference_stage import IdentityInferenceStage

from morpheus.pipeline.pass_thru_type_mixin import PassThruTypeMixin
from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.pipeline.single_port_stage import SinglePortStage

from morpheus.messages import ControlMessage
from morpheus.messages import TensorMemory

from morpheus.utils.logger import configure_logging, reset_logging

import mrc
from mrc.core import operators as ops

---

## Identity Inference Stage

We will be looking at actual inference stages later in the workshop, but for now we are going to utilize a pre-built Morpheus stage, primarily used for testing: `morpheus.stages.inference.identity_inference_stage.IdentityInferenceStage`.

`IdentityInferenceStage` expects messages of the `ControlMessage` type, and expects that incoming control messages contain a `seq_ids` tensor.

In service of preparing control messages for the `IdentityInferenceStage`, we are going to write a custom `AddTensors` stage which expects a control message, adds to it a `"seq_ids"` tensor, and then emits the control message to the next stage of the pipeline, ostensibly for inference.

For the sake of giving you an intuition about sequence IDs in the previous notebook we discussed why and how they are actually created. For our purposes here while using our no-op inference stage, we only need our custom stage to create a `"seq_ids"` tensor with the correct count.

In [ ]:
class AddTensors(PassThruTypeMixin, GpuAndCpuMixin, SinglePortStage):

    @property
    def name(self) -> str:
        return "add-tensors"

    def accepted_types(self) -> tuple:
        return (ControlMessage, )

    def supports_cpp_node(self) -> bool:
        return False

    def on_data(self, message: ControlMessage) -> ControlMessage:
        
        # Here, we collect attributes for our simulated inference tensor data
        count, ncols = message.payload().get_data().shape
        
        # Here we create fake seq_ids for our no-op inference following the setup strategy from the previous notebook.
        seq_ids = cp.zeros((count, 3), dtype=cp.uint32)
        seq_ids[:, 0] = cp.arange(0, count, dtype=cp.uint32)        
        
        # Here we create some fake data for our no-op inference
        data = cp.zeros(shape=(count, ncols), dtype=cp.float32)
        
        # Initialize our TensorMemory object
        tensor_memory = TensorMemory(count=count)
        # add our tensors to the TensorMemory object
        tensor_memory.set_tensor("seq_ids", seq_ids)
        tensor_memory.set_tensor("input__0", data)

        # Set the tensor memory in the ControlMessage
        message.tensors(tensor_memory)

        return message

    def _build_single(self, builder: mrc.Builder, input_node: mrc.SegmentObject) -> mrc.SegmentObject:
        node = builder.make_node(self.unique_name, ops.map(self.on_data))
        builder.make_edge(input_node, node)

        return node

---

## Create a No-op Inference Pipeline

Using our simple user auth logs data, we'll now create a pipeline that uses both our custom `AddTensor` stage and the `IdentityInferenceStage`.

In [ ]:
input_file = 'data/simple_user_log.jsonlines'

In [ ]:
config = Config(feature_length=6)

In [ ]:
pipeline = LinearPipeline(config)

pipeline.set_source(FileSourceStage(config, filename=input_file, iterative=False, repeat=100))
pipeline.add_stage(DeserializeStage(config))

pipeline.add_stage(AddTensors(config))

pipeline.add_stage(IdentityInferenceStage(config))

in_mem_sink = pipeline.add_stage(InMemorySinkStage(config))

pipeline.add_stage(MonitorStage(config, description="Pipeline throughput"))

In [ ]:
pipeline.build()

In [ ]:
viz_file = './pipeline_visualizations/no-op-inference.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

---

## Run the No-op Inference Pipeline

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
# Run the pipeline
await pipeline.run_async()

In [ ]:
control_message = in_mem_sink.get_messages()[0]

Since our pipeline did not perform any work on our message payload dataframe, we see it unchanged after passing through the pipeline.

In [ ]:
control_message.payload().get_data()

Let's take a look at the tensors attached to our control message after it has passed through the pipeline.

In [ ]:
tensors = control_message.tensors().get_tensors()

Here we see a single `probs` tensor that the `IdentityInferenceStage` has created.

In [ ]:
tensors

In the case of our simple user authentication log data, which contains 10 rows, the no-op inference stage has created 10 tensors, one for each row of data, with 6 scalar values each. This number matches the `feature_length` setting in our configuration object.

In [ ]:
post_inference_tensors = tensors["probs"]

In [ ]:
post_inference_tensors.shape

When performing actual inference, we would expect these 256 floating point values for each row in the data to contain the actual results of the inference, which could vary widely in their signficance depending on the kind of machine learning inference we were performing.